In [19]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, message=".*escape sequence.*")
import pandas as pd
import httpx
from openai import OpenAI
import time
import os
from dotenv import load_dotenv, find_dotenv

os.environ["OPENAI_API_KEY"] = 'hey' # for importing trialmind, api_key should be set beforehand
os.environ['TEMPERATURE'] = '0'
load_dotenv(find_dotenv(usecwd=True), override=True)
use = 'cpp'

if use == 'hf':
    os.environ["BASE_URL"] = "https://router.huggingface.co/v1"
    os.environ["MODEL_NAME"] ='Qwen/Qwen3-14B'
    os.environ["OPENAI_API_KEY"] = os.getenv("HUGGINGFACE_HUB_TOKEN2")
elif use == 'olm':
    os.environ["BASE_URL"] = "http://localhost:11434/v1"
    os.environ["MODEL_NAME"] ='qwen3:14b'
    os.environ["OPENAI_API_KEY"] = 'hey'
elif use == 'cpp':
    os.environ["BASE_URL"] = "http://localhost:8080/v1"
    os.environ["MODEL_NAME"] ='unsloth/Qwen3-14B'#'ggml-org/gemma-3-1b-it-GGUF'
    os.environ["OPENAI_API_KEY"] = 'hey'
    
import report_gen
                      
%load_ext autoreload
%autoreload 2

ValueError: Unknown scheme for proxy URL URL('socks4://127.0.0.1:10808')

In [13]:
!pip install -U requests[socks]

ERROR: Could not install packages due to an OSError: Missing dependencies for SOCKS support.



In [2]:
client = OpenAI(
            base_url=os.getenv("BASE_URL"),
            api_key=os.getenv("OPENAI_API_KEY"),
            http_client=httpx.Client(verify=False)
        )

os.environ["MODEL_NAME"] = 'unsloth/Qwen3-14B'
start = time.time()
response = client.chat.completions.create(
        model=os.getenv("MODEL_NAME"),
        messages=[{'role':'user','content':'hey'}],  # Ensure messages is a list
        temperature=0,
    extra_body={"reasoning_effort": "none"}
    )
end = time.time()
print(end - start)

response#.choices[0]#.message.content

19.113533973693848


ChatCompletion(id='chatcmpl-ZNUTGelA8TxpMvhtHP148eH6e1f6mL14', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='<think>\nOkay, the user just said "hey". That\'s pretty open-ended. I need to figure out the best way to respond. Maybe they\'re testing the waters or just starting a conversation. I should keep it friendly and open. Let me ask how I can assist them. That way, I can tailor my response to their needs. I\'ll make sure to use a welcoming tone and offer help with whatever they need. Let me check if there\'s any specific context I should consider, but since there\'s none, a general offer should work. Alright, time to respond.\n</think>\n\nHello! How can I assist you today? 😊', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1777893969, model='unsloth/Qwen3-14B', object='chat.completion', service_tier=None, system_fingerprint='b8864-ff6b1062a', usage=CompletionUsage(com

## preloading; checking

In [3]:
import logging
logger = logging.getLogger('').setLevel(logging.INFO)

logger = logging.getLogger('my_module_name').setLevel(logging.INFO)

# Silence other loggers
for log_name, log_obj in logging.Logger.manager.loggerDict.items():
    if log_name != 'my_module_name':
        log_obj.disabled = True
logger = logging.getLogger().setLevel(logging.INFO)    
logging.info('hey')

INFO:root:hey


In [4]:
import extract

treatments_eng=\
[  'Crizotinib',   'Lorlatinib',  'Necitumumab',  'Nimotuzumab',
  'Panitumumab',    'Cetuximab',   'Brigatinib',  'Ramucirumab',
 'Pomalidomide',   'Toremifene',    'Denosumab',    'Duvelisib',
   'Inavolisib',  'Bevacizumab',  'Anastrozole',    'Letrozole',
    'Tamoxifen',  'Fulvestrant',   'Exemestane',    'Buserelin']

fin_condition = 'Colorectal cancer'
file_path = "docs/CC_215_L00_DL_fusion_fixed_oncobox_ru.pdf"
#fin_condition,treatments_eng,df2,fin_condition_ru,treatments_ru = extract.info_from_doc(file_path, with_ru=True)
    
print(f'{fin_condition}, {treatments_eng[:20]}')

Colorectal cancer, ['Crizotinib', 'Lorlatinib', 'Necitumumab', 'Nimotuzumab', 'Panitumumab', 'Cetuximab', 'Brigatinib', 'Ramucirumab', 'Pomalidomide', 'Toremifene', 'Denosumab', 'Duvelisib', 'Inavolisib', 'Bevacizumab', 'Anastrozole', 'Letrozole', 'Tamoxifen', 'Fulvestrant', 'Exemestane', 'Buserelin']


In [5]:
treatments_eng=\
[  'Crizotinib',   'Lorlatinib',  'Necitumumab',  'Nimotuzumab',
  'Panitumumab',    'Cetuximab',   'Brigatinib',  'Ramucirumab',
 'Pomalidomide',   'Toremifene',    'Denosumab',    'Duvelisib',
   'Inavolisib',  'Bevacizumab',  'Anastrozole',    'Letrozole',
    'Tamoxifen',  'Fulvestrant',   'Exemestane',    'Buserelin']
treatments_eng[7:8]

['Ramucirumab']

## generating a report

In [ ]:
treatments_eng, fin_condition, fin_condition_ru = report_gen.get_res(
            db_name='hybrid_fin_new',
            file_path = "docs/CC_215_L00_DL_fusion_fixed_oncobox_ru.pdf", # path to an oncobox report
            model_translate='unsloth/Qwen3-14B',#'unsloth/Qwen3-14B-GGUF', # model to use for translation
            model_main='unsloth/Qwen3-14B',#'unsloth/Qwen3-14B-GGUF', # model to use for screening, extracting results.
            n_papers=0, # number of papers to screen
            ct_pages=10, # number of pages of clinical trials to screen
            # criteria for screening clinical trials
            ct_criteria = \
            ["Does the trial focus on patients with '{fin_condition}'?",
             "Does the trial examine the use or sensitivity of '{treatment}' among main treatments?"],
            # criteria for screening papers
            papers_criteria=\
            ["Does the study focus on patients/models/cells with '{fin_condition}'?",
             "Does the study examine the use/effect/sensitivity of '{treatment}' among main treatments?", 
             "Does the study describe the effect of '{treatment}' treatment?"],
            fields=\
            ['{treatment} effectiveness | string | The outcome of treating {fin_condition} with {treatment}',
             "Type of study | string | Is the study in vitro, in vivo, clinical trial or others",
             "Num participants | string | How many patients with {fin_condition} are treated with {treatment} in the study"],
            # to pass screening, a clinical trial must have a score => ct_screen_thresh for each criterium
            ct_screen_thresh=[1,0], 
            # to pass screening, a paper must have a score => paper_screen_thresh for each criterium
            paper_screen_thresh = [1,1,0],
            save_files=True, # whether to save screening and extracted results in .csv
            n_treats = 20, # how many treatments to analyse from the report
           )

INFO:root:unsloth/Qwen3-14B
INFO:root:GETTING INFO FROM FILE
INFO:root:http://localhost:8080/v1
INFO:root:http://localhost:8080/v1
INFO:root:Colorectal cancer, <StringArray>
[  'Crizotinib',   'Lorlatinib',  'Necitumumab',  'Nimotuzumab',
  'Panitumumab',    'Cetuximab',   'Brigatinib',  'Ramucirumab',
 'Pomalidomide',   'Toremifene',    'Denosumab',    'Duvelisib',
   'Inavolisib',  'Bevacizumab',  'Anastrozole',    'Letrozole',
    'Tamoxifen',  'Fulvestrant',   'Exemestane',    'Buserelin']
Length: 20, dtype: str
INFO:root:4.8643035888671875
INFO:root:Crizotinib
INFO:root:exception:  cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:cannot close client cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:Lorlatinib
INFO:root:
GETTING CLINICAL TRIALS for Lorlatinib
INFO:root:(0, 1)
INFO:root:exception:  cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:cannot close cli

NCT02394795
6
6253.0
NCT01412957
6
8385.333333333334
3514.3333333333335
3570.0
3656.0
3581.0
3658.6666666666665
3674.6666666666665
NCT01412957  with several: 7
NCT00891930
9
6923.333333333333
NCT03442569
4
4359.333333333333
NCT00339183
4
7336.0
NCT02448810
1
5067.333333333333
NCT00083616
3
4173.666666666667
NCT00411450
3
5472.333333333333
NCT03983993
4
4150.0
NCT00655499
3
4821.0
NCT02613221
6
5625.333333333333
NCT01226719
3
4147.666666666667
NCT00089635
3
4202.333333333333
NCT00940316
3
5539.0
NCT04117945
6
6652.0
NCT00364013
4
7493.333333333333
NCT00630786
3
4886.0
NCT03263429
2
3799.3333333333335
NCT00508404
7
8358.666666666666
3655.3333333333335
3372.6666666666665
3457.6666666666665
3445.6666666666665
3497.0
3427.0
3426.6666666666665
NCT00508404  with several: 8
NCT00418938
5
5639.333333333333
NCT01732783
1
3277.0
NCT00111761
2
3567.6666666666665
NCT02508077
1
3146.0
NCT01504477
2
3306.0
NCT00788957
7
7392.0
NCT00332163
6
8104.0
NCT00115765
14
9056.333333333334
3170.0
3167.0
2963.6

In [25]:
406/60

6.766666666666667

In [10]:
treatments_eng, fin_condition, fin_condition_ru = report_gen.get_res(
            db_name='hybrid_fin_new',
            file_path = "docs/CC_215_L00_DL_fusion_fixed_oncobox_ru.pdf", # path to an oncobox report
            model_translate='unsloth/Qwen3-14B',#'unsloth/Qwen3-14B-GGUF', # model to use for translation
            model_main='unsloth/Qwen3-14B',#'unsloth/Qwen3-14B-GGUF', # model to use for screening, extracting results.
            n_papers=0, # number of papers to screen
            ct_pages=10, # number of pages of clinical trials to screen
            # criteria for screening clinical trials
            ct_criteria = \
            ["Does the trial focus on patients with '{fin_condition}'?",
             "Does the trial examine the use or sensitivity of '{treatment}' among main treatments?"],
            # criteria for screening papers
            papers_criteria=\
            ["Does the study focus on patients/models/cells with '{fin_condition}'?",
             "Does the study examine the use/effect/sensitivity of '{treatment}' among main treatments?", 
             "Does the study describe the effect of '{treatment}' treatment?"],
            fields=\
            ['{treatment} effectiveness | string | The outcome of treating {fin_condition} with {treatment}',
             "Type of study | string | Is the study in vitro, in vivo, clinical trial or others",
             "Num participants | string | How many patients with {fin_condition} are treated with {treatment} in the study"],
            # to pass screening, a clinical trial must have a score => ct_screen_thresh for each criterium
            ct_screen_thresh=[1,0], 
            # to pass screening, a paper must have a score => paper_screen_thresh for each criterium
            paper_screen_thresh = [1,1,0],
            save_files=True, # whether to save screening and extracted results in .csv
            n_treats = 20, # how many treatments to analyse from the report
           )

INFO:root:unsloth/Qwen3-14B
INFO:root:GETTING INFO FROM FILE
INFO:root:http://localhost:8080/v1
INFO:root:http://localhost:8080/v1
INFO:root:Colorectal cancer, <StringArray>
[  'Crizotinib',   'Lorlatinib',  'Necitumumab',  'Nimotuzumab',
  'Panitumumab',    'Cetuximab',   'Brigatinib',  'Ramucirumab',
 'Pomalidomide',   'Toremifene',    'Denosumab',    'Duvelisib',
   'Inavolisib',  'Bevacizumab',  'Anastrozole',    'Letrozole',
    'Tamoxifen',  'Fulvestrant',   'Exemestane',    'Buserelin']
Length: 20, dtype: str
INFO:root:5.180840969085693
INFO:root:Crizotinib
INFO:root:
GETTING CLINICAL TRIALS for Crizotinib
INFO:root:(1, 10)
INFO:root:0.29787659645080566
INFO:root:SCREENING CLINICAL TRIALS
INFO:root:exception:  CTScreening.run() got an unexpected keyword argument 'population'
INFO:root:cannot close client cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:Lorlatinib
INFO:root:
GETTING CLINICAL TRIALS for Lorlatinib
INFO:root:(0, 1)
INFO:root

NCT02394795
6
6253.0
NCT01412957
5
7525.0
NCT00891930
9
6923.333333333333
NCT03442569
3
4021.3333333333335
NCT00339183
3
6141.666666666667
NCT02448810
1
5067.333333333333
NCT00083616
2
3788.6666666666665
NCT00411450
3
5472.333333333333
NCT03983993
4
4150.0
NCT00655499
3
4821.0
NCT02613221
6
5625.333333333333
NCT01226719
3
4147.666666666667
NCT00089635
2
3796.0
NCT00940316
3
5539.0
NCT04117945
6
6652.0
NCT00364013
4
7493.333333333333
NCT00630786
2
4321.666666666667
NCT03263429
2
3799.3333333333335
NCT00508404
7
8358.666666666666
3655.3333333333335
3372.6666666666665
3457.6666666666665
3445.6666666666665
3497.0
3427.0
3426.6666666666665
NCT00508404  with several: 8
NCT00418938
4
5069.333333333333
NCT01732783
1
3277.0
NCT00111761
2
3567.6666666666665
NCT02508077
1
3146.0
NCT01504477
2
3306.0
NCT00788957
7
7392.0
NCT00332163
5
7307.333333333333
NCT00115765
14
9056.333333333334
3170.0
3167.0
2963.6666666666665
3045.6666666666665
3145.6666666666665
3016.6666666666665
3201.0
3198.0
3190.33333

INFO:root:exception:  RetryError[<Future at 0x234587bcbd0 state=finished raised APIConnectionError>]
INFO:root:cannot close client cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:Cetuximab
INFO:root:
GETTING CLINICAL TRIALS for Cetuximab
INFO:root:(87, 10)
INFO:root:426.2098948955536
INFO:root:SCREENING CLINICAL TRIALS
INFO:root:exception:  CTScreening.run() got an unexpected keyword argument 'population'
INFO:root:cannot close client cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:Brigatinib
INFO:root:
GETTING CLINICAL TRIALS for Brigatinib
INFO:root:(0, 1)
INFO:root:exception:  cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:cannot close client cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:Ramucirumab
INFO:root:exception:  cannot access local variable 'clientqh' where it is not associated with a value
INFO:root:canno

In [19]:
d['nctId']

0    NCT01286818
1    NCT01183780
2    NCT01111604
3    NCT01079780
4    NCT00862784
Name: nctId, dtype: str

In [17]:
d = pd.read_csv('res_files/Colorectal_cancer/ctrials_res_df_Ramucirumab.csv')
d['res'].values


<StringArray>
[                                                                                                                                                                               '{"reasoning":"The values were calculated from the group 'FOLFIRI Plus Ramucirumab (IMC-1121B)' with a population of 6 participants. The percentages were calculated using the formula (number_participants * 100) / population.","complete_response":0.0,"partial_response":16.67,"objective_response_rate":16.67,"stable_disease":66.67,"disease_control_rate":83.33,"progression_free_survival":0.0,"overall_survival":0.0}',
                                                                                 '{"reasoning":"The values were calculated from the group 'Ramucirumab + FOLFIRI' (OG000) as it is the main group with Ramucirumab. The objective response rate (ORR) is provided as 13.4% for this group. Since the other required values (CR, PR, SD, DCR, PFS, OS) are not explicitly provided in the input, they are 

In [ ]:
Brigatinib

In [ ]:
#Ramucirumab no 'results' during rerank

In [ ]:
treatments_eng, fin_condition, fin_condition_ru = report_gen.get_res(
            db_name='hybrid_fin_new',
            file_path = "docs/CC_215_L00_DL_fusion_fixed_oncobox_ru.pdf", # path to an oncobox report
            model_translate='unsloth/Qwen3-14B',#'unsloth/Qwen3-14B-GGUF', # model to use for translation
            model_main='unsloth/Qwen3-14B',#'unsloth/Qwen3-14B-GGUF', # model to use for screening, extracting results.
            n_papers=0, # number of papers to screen
            ct_pages=10, # number of pages of clinical trials to screen
            # criteria for screening clinical trials
            ct_criteria = \
            ["Does the trial focus on patients with '{fin_condition}'?",
             "Does the trial examine the use or sensitivity of '{treatment}' among main treatments?"],
            # criteria for screening papers
            papers_criteria=\
            ["Does the study focus on patients/models/cells with '{fin_condition}'?",
             "Does the study examine the use/effect/sensitivity of '{treatment}' among main treatments?", 
             "Does the study describe the effect of '{treatment}' treatment?"],
            fields=\
            ['{treatment} effectiveness | string | The outcome of treating {fin_condition} with {treatment}',
             "Type of study | string | Is the study in vitro, in vivo, clinical trial or others",
             "Num participants | string | How many patients with {fin_condition} are treated with {treatment} in the study"],
            # to pass screening, a clinical trial must have a score => ct_screen_thresh for each criterium
            ct_screen_thresh=[1,0], 
            # to pass screening, a paper must have a score => paper_screen_thresh for each criterium
            paper_screen_thresh = [1,1,0],
            save_files=True, # whether to save screening and extracted results in .csv
            n_treats = 20, # how many treatments to analyse from the report
           )

INFO:root:unsloth/Qwen3-14B
INFO:root:GETTING INFO FROM FILE
C:\Users\105\GitHub\VirtualOncologist\extract.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[df.score.isin(['- ','-',' -']),'score'] = '10.00'
C:\Users\105\GitHub\VirtualOncologist\extract.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.score = df.score.astype(float)
C:\Users\105\GitHub\VirtualOncologist\extract.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats i

http://localhost:8080/v1
http://localhost:8080/v1


INFO:root:Colorectal cancer, ['Crizotinib' 'Lorlatinib' 'Necitumumab' 'Nimotuzumab' 'Panitumumab'
 'Cetuximab' 'Brigatinib' 'Ramucirumab' 'Pomalidomide' 'Toremifene'
 'Denosumab' 'Duvelisib' 'Inavolisib' 'Bevacizumab' 'Anastrozole'
 'Letrozole' 'Tamoxifen' 'Fulvestrant' 'Exemestane' 'Buserelin']
INFO:root:11.809432983398438
INFO:root:using existing db
INFO:root:vector store NAME: hybrid_one_db_Ramucirumab
INFO:root:
GETTING CLINICAL TRIALS for Ramucirumab
INFO:root:(6, 10)
INFO:root:10.876403331756592
INFO:root:SCREENING CLINICAL TRIALS
INFO:root:parsed_results in asynch: [PaperEvaluation(evaluations=['YES', 'YES'], rationale=['The trial specifically mentions Japanese participants with advanced colorectal carcinoma (CRC).', 'The trial investigates the safety and tolerability of Ramucirumab in combination with FOLFIRI, indicating that Ramucirumab is a main treatment being examined.'])]
INFO:root:parsed_results in asynch: [PaperEvaluation(evaluations=['YES', 'NO'], rationale=['The trial 

int() argument must be a string, a bytes-like object or a real number, not 'NoneType'


INFO:root:using existing db
INFO:root:vector store NAME: hybrid_one_db_Pomalidomide
INFO:root:
GETTING CLINICAL TRIALS for Pomalidomide
INFO:root:(0, 1)
INFO:root:using existing db
INFO:root:vector store NAME: hybrid_one_db_Toremifene
INFO:root:
GETTING CLINICAL TRIALS for Toremifene
INFO:root:(0, 1)
INFO:root:using existing db
INFO:root:vector store NAME: hybrid_one_db_Denosumab
INFO:root:
GETTING CLINICAL TRIALS for Denosumab
INFO:root:(2, 10)
INFO:root:15.354169845581055
INFO:root:SCREENING CLINICAL TRIALS
INFO:root:parsed_results in asynch: [PaperEvaluation(evaluations=['NO', 'YES'], rationale=['The trial focuses on patients with breast cancer, not colorectal cancer.', 'The trial examines the use of denosumab as a treatment in patients receiving aromatase inhibitor therapy.'])]
INFO:root:parsed_results in asynch: [PaperEvaluation(evaluations=['YES', 'NO'], rationale=['The trial focuses on patients with colorectal cancer (CRC).', 'The trial examines RXC004 and its combination with n

In [ ]:
Ramucirumab clinical

In [ ]:
'''
Panitumumab 
INFO:root:126
INFO:root:7806.037273406982
INFO:root:GETTING FULL TEXT PAPERS

INFO:root:papers embedded and new: 126/126
INFO:root:FIN EMBEDDING
INFO:root:7810.858558177948

// INFO:root:35137.4686229229

Cetuximab 150
INFO:root:132
INFO:root:7107.122780323029
INFO:root:GETTING FULL TEXT PAPERS
'''

## if ctrial res >= context_size

In [6]:
from pydantic import BaseModel, Field, field_validator # This is the new version
from typing import TypeVar
from deepmerge import always_merger

class Measure(BaseModel):
    measure_description: str = Field(description='Short description of what is being measured; UNDER 200 characters',
                                    max_length=200)
    measure_result: float = Field(description='Percent of participants')

    @field_validator('measure_description', mode='before')
    @classmethod
    def truncate_string(cls, v):
        return v[:200]
    
class GroupMeasures(BaseModel):
    group_description: str = Field(description='Short group description; UNDER 200 characters',
                                  max_length=200)
    measures: list[Measure]

    @field_validator('group_description', mode='before')
    @classmethod
    def truncate_string(cls, v):
        return v[:200]
    
class Outcome(BaseModel):
    description: str = Field(description='Short description of an outcome; UNDER 200 characters',
                            max_length=200)
    population: int = Field(description='Total number of participants.')
    time_frame: str = Field(description='Time frame', max_length=200 )
    measures: list[GroupMeasures]

    @field_validator('description', mode='before')
    @classmethod
    def truncate_string(cls, v):
        return v[:200]

class Outcomes(BaseModel):
    outcomes: list[Outcome]

In [ ]:
Ramucirumab 

In [7]:
import pandas as pd
import numpy as np
import extract
import ast
all_studies = pd.read_csv('res_files/Colorectal_cancer/ctrials_all_df_Ramucirumab.csv'
)
all_studies['outcomeMeasures'] = all_studies['outcomeMeasures'
                                        ].apply(lambda x: ast.literal_eval(x))

all_studies['res_with_part'] = all_studies['outcomeMeasures'
                ].apply(lambda x: [obj for obj in x if(obj.get('unitOfMeasure',
                                                               '').lower() in \
                                                       ['percentage of participants',
                                                        'participants'])])
all_studies.outcomeMeasures.str.len()

0     9
1    31
2     7
3    15
4     4
5    17
Name: outcomeMeasures, dtype: int64

In [9]:
ctrials_res(ec_pred, chosen, ct_screen_thresh)

NameError: name 'ec_pred' is not defined

In [34]:
len(all_studies.res_with_part.values[0]), len(str(all_studies.res_with_part.values[0]))

(3, 16619)

In [8]:
def ctrials_res(ec_pred, chosen, ct_screen_thresh):
    
    evals = [i for i in chosen[['s1','s2']].values]
    word2int = {"YES": 1, "UNCERTAIN": 0,"NO": -1}
    new_evals = []
    for one_e in evals:
        new_evals.append([word2int.get(item, 0) for item in one_e ])
    new_evals = np.array(new_evals)    
    chosen[['s1','s2']] = new_evals
    
    PROMPT_RES_EXTRACTION  = f'''
    You are a clinical specialist analyzing clinical trial study reports. 
    Your task is to to extract specific information as structured data.

    # Reply Format: 
    Return the information in the following JSON-format.
    ```
    {Outcomes.model_json_schema()}
    ```
    You MUST return ONLY valid JSON, Do NOT include any explanations, comments, or extra text.
    Descriptions MUST be under 200 characters; summarize if needed. 
    '''
    
    at_once = False

    
    to_work = chosen[chosen.res_with_part.str.len()>0]
    resultsct = {}
    if to_work.shape[0]:
        openai_client = OpenAI(
            base_url=os.getenv("BASE_URL"),
            api_key=os.getenv("OPENAI_API_KEY"),
            http_client=httpx.Client(verify=False)
        )
        messages = []
        ctx_size=8192#int(os.getenv("CONTEXT_SIZE"))
        for ct_results, ct_id in zip(to_work.res_with_part.values, 
                              to_work.nctId.values):
            # if results for one ctrial are greater than context_size
            if len(str(ct_results)) >= (ctx_size - len(PROMPT_RES_EXTRACTION)):
                # check if each result is greater
                for one_ct_res in ct_results:
                    # just skip; todo: cut?
                    if len(str(one_ct_res)) >= (ctx_size - len(PROMPT_RES_EXTRACTION)):
                        print('skip', len(str(one_ct_res)))
                        pass
                    else:
                        print('save', len(str(one_ct_res)))
                        messages.append([{'ct_id':ct_id},
                                         {'role':'system', 
                                          'content':PROMPT_RES_EXTRACTION+' \no_think'},
                                         {'role':'user', 
                                          'content':f"{one_ct_res}"}])
                
                
            messages.append([{'ct_id':ct_id},
                             {'role':'system', 
                              'content':PROMPT_RES_EXTRACTION+' \no_think'},
                             {'role':'user', 
                              'content':f"{ct_results}"}])
        if at_once:
            response = batch_call_openai(messages, os.getenv('MODEL_NAME'), 
                              int(os.getenv('TEMPERATURE')), thinking=False)
        else:
            #print(messages)
            last_id = ''
            for message in messages:
                print(message[0])
                response = openai_client.chat.completions.parse(
                    model=os.getenv('MODEL_NAME'),
                    messages=message[1:], # skipping 'ct_id' info
                    temperature=0,
                    response_format=Outcomes,
                    extra_body={"reasoning_effort": "none"}
                )
                fin = response.choices[0].message.parsed
                # appending to existing ct_id, if it exists
                if message[0]['ct_id']==last_id:
                    resultsct[message[0]['ct_id']].append(fin)
                else:
                    resultsct[message[0]['ct_id']] = [fin]
                last_id = message[0]['ct_id']
                #answer = fin.strip('<think>\n\n</think>\n\n')
                #resultsct.append(fin)

        # filling df with extracted results
        to_work['res']=''
        for ct_id in to_work.nctId.values:
            ct_results = resultsct.get(ct_id,'')
            print(len(ct_results))
            if ct_results:
                if len(ct_results)>1:
                    zero_part = ct_results[0]
                    for one_part in ct_results[1:]:
                        zero_part = merge_pydantic_models(zero_part, one_part)
                        
                    to_work.loc[to_work.nctId==ct_id, 'res'] = zero_part.model_dump_json()
                else:
                    to_work[to_work.nctId==ct_id]['res'] = ct_results[0].model_dump_json()
    
    return resultsct, to_work

In [ ]:
'''
engthFinishReasonError: Could not parse response content as the length 
limit was reached - CompletionUsage(completion_tokens=971, 
prompt_tokens=7221, total_tokens=8192, 
completion_tokens_details=None, 
prompt_tokens_details=PromptTokensDetails(audio_tokens=None, 
cached_tokens=7220))
'''

In [8]:
from trialmind.api import StudyCharacteristicsExtraction, LiteratureScreening, CTScreening
import extract
import numpy as np

ct_pages=2
ct_criteria = \
            ["Does the trial focus on patients with '{fin_condition}'?",
             "Does the trial examine the use or sensitivity of '{treatment}' among main treatments?"]
ct_screen_thresh=[1,0]
treatment = 'Panitumumab'
fin_condition = 'Colorectal cancer'

all_studies = extract.get_clinicaltrials(f'''"{fin_condition}"''', 
                                                  #' OR '.join(treatments_eng), 
                                                   treatment,  
                                                  max_studies=ct_pages).iloc[:2]
print(all_studies.shape)

# if there are any clinical trials
if (all_studies.shape[0]):
    if ('briefTitle' in all_studies.columns) & ('briefSummary' in all_studies.columns):
        all_text = all_studies.briefTitle.fillna("") + ": " + all_studies.briefSummary.fillna("")
    else:
        all_text = pd.Series([""]*all_studies.shape[0])
    api = CTScreening()
    
    logging.info('SCREENING CLINICAL TRIALS')
    ec_pred = api.run(
        population = f"Patients with {fin_condition} undergoing treatment with {treatment}",
        intervention = f"{treatment}",
        comparator = "",
        outcome = "",
        llm = os.getenv("MODEL_NAME"),
        criteria = [i.format(fin_condition=fin_condition,
                             treatment=treatment) for i in ct_criteria],
        papers = all_text.values.tolist(), # make for the top-100 for demo
    )
    all_studies[['s1','s2']] = [i.evaluations for i in ec_pred]
    all_studies[['r1','r2']] = [i.rationale for i in ec_pred]


INFO:root:SCREENING CLINICAL TRIALS


(2, 10)


INFO:root:parsed_results in asynch: [PaperEvaluation(evaluations=['YES', 'YES'], rationale=['The trial explicitly focuses on patients with colorectal cancer (CRC), specifically those with BRAF-mutant V600E advanced or metastatic CRC.', 'The trial examines the use of panitumumab (P) in combination with other agents (dabrafenib and/or trametinib) in patients with CRC.'])]
INFO:root:parsed_results in asynch: [PaperEvaluation(evaluations=['YES', 'YES'], rationale=['The trial focuses on patients with metastatic colorectal cancer.', 'The trial examines the use of Panitumumab in combination with FOLFOXIRI as a first-line treatment.'])]


## back

In [10]:
#2071s full time for Lorlatinib
# 1504 s for Necitumumab
# 4354s full time Nimotuzumab (822s to embed 10 docs..)
# 1178s to screen 46 cts for panitumumab 

In [7]:
2071/60

34.516666666666666

In [ ]:
'''
without clinical trials:
56 sec to get INFO FROM FILE (3 sec after using table to translate)
\\ 5 sec to get 67 paper ids
37 sec to screen 2 papers locally (32 sec for 5 papers with HF)\\1000s to screen 67 papers locally
\\ 46 sec to get fulltext for 39papers
4.7 sec to embed 1 paper (23 sec for 3 papers)\\225s embed 39 papers
45.6 sec to exract results for 3 fields for 1 paper (439sec for 3 papers with HF!) \\ 1859s - 3fields-39papers
... (Full time 525sec) \\ full time 2145sec without cts, screen and embedding
'''

In [6]:
fin_condition, len(treatments_eng), treatments_eng[:3]

('Colorectal cancer',
 77,
 <StringArray>
 ['Crizotinib', 'Lorlatinib', 'Necitumumab']
 Length: 3, dtype: str)

In [161]:
# open-label single arm phase i trial -> открытом, одноармий фазе I испытания ??
report_gen.fill_pdf(treatments_eng, # list of extracted treatment names
                    fin_condition, # the condition 
                    fin_condition_ru,
                    model_translate='Qwen/Qwen3-14B', # model to use for translation
                    n_treats=1,
                    path_file='fin_reports/', 
                    fields=fields
                   )

<source id="0"><result id="0">Combination binimetinib/crizotinib showed poor tolerability with no objective responses observed in RASMT advanced CRC patients.</result><context id="0">[. /n/n '. /n/n M</context><result id="1">open-label, single arm phase I trial</result><context id="1">[. /n/n '. /n/n i</context><result id="2">36 patients with advanced RASMT CRC</result><context id="2">[. /n/n '. /n/n p</context></source>

<source id="1"><result id="0">Crizotinib was more sensitive to low-risk group in CRC treatment.</result><context id="0">[. /n/n '. /n/n a</context><result id="1">clinical trial</result><context id="1">[. /n/n '. /n/n o</context></source>

<source id="2"><result id="0">Crizotinib selectively inhibited growth of ARID1A-deficient CRC cells in vitro and xenograft models, showing synthetic lethality.</result><context id="0">[. /n/n '. /n/n a</context><result id="1">The study includes both in vitro and in vivo experiments.</result><context id="1">[. /n/n '. /n/n i</context>

INFO:root:23.855294227600098
